# M2.2.e: 169-feature integration + LightGBM val AUC

Integrates all M2.2 sub-steps into a single pipeline and runs LightGBM.

Pipeline: cloud_coverage fix → ClusterNo merge → sort → value-change (120)
→ SavGol residual → dayofyear → downsampling → CV split → StandardScaler → LightGBM

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.signal import savgol_filter
import lightgbm as lgb

train = pd.read_csv("../data/raw/train_features.csv")
print(f"Loaded: {train.shape}")

# M2.2.0: cloud_coverage sentinel fix
train["cloud_coverage"] = train["cloud_coverage"].replace({255: 10})
print("cloud_coverage fix applied")

Loaded: (1749494, 57)
cloud_coverage fix applied


In [2]:
clusterno_df = pd.read_csv("../data/interim/clusterno.csv")
train = train.merge(clusterno_df, on="building_id", how="left")
nan_clusterno = train["ClusterNo"].isna().sum()
print(f"After ClusterNo merge: {train.shape}")
print(f"ClusterNo NaN: {nan_clusterno}")
assert nan_clusterno == 0, f"Unexpected ClusterNo NaN: {nan_clusterno}"

After ClusterNo merge: (1749494, 58)
ClusterNo NaN: 0


In [3]:
train = train.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

shifts = (
    list(np.arange(-24, 0))
    + list(np.arange(1, 25))
    + list(np.arange(-168, -24, 24))
    + list(np.arange(48, 169, 24))
)

for n in shifts:
    train[f"lag_value_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) - train["meter_reading"]
    )
    train[f"lag_value_ratio_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) + 1
    ) / (train["meter_reading"] + 1)

lag_cols = [c for c in train.columns if c.startswith("lag_value_")]
print(f"Lag features: {len(lag_cols)}")
print(f"Train shape: {train.shape}")
assert len(lag_cols) == 120, f"Expected 120, got {len(lag_cols)}"

C:\Users\tonykuo\AppData\Local\Temp\ipykernel_34864\182555182.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[f"lag_value_{n}"] = (
C:\Users\tonykuo\AppData\Local\Temp\ipykernel_34864\182555182.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[f"lag_value_ratio_{n}"] = (


Lag features: 120
Train shape: (1749494, 178)


In [4]:
results = []
for bid in train["building_id"].unique():
    tmp = train[train["building_id"] == bid].copy()
    smoothed = savgol_filter(tmp["meter_reading"].ffill().bfill().fillna(0), 5, 3)
    tmp["Residual_savgol_w5p3"] = tmp["meter_reading"] - smoothed
    results.append(tmp)
train = pd.concat(results).sort_index()
print(f"After SavGol: {train.shape}")

After SavGol: (1749494, 179)


In [5]:
train["dayofyear"] = (
    pd.to_datetime(train["timestamp"]).dt.dayofyear
    + pd.to_datetime(train["timestamp"]).dt.hour / 24
)
print(f"After dayofyear: {train.shape}")

After dayofyear: (1749494, 180)


In [6]:
X_full = train.select_dtypes(
    include=["float", "int", "int64", "int32", "float64", "float32"]
)
drop_cols = ["anomaly", "wind_direction", "air_temperature_std_lag73"]
X_full = X_full.drop(columns=[c for c in drop_cols if c in X_full.columns])

print(f"Feature count: {X_full.shape[1]}")
print("Expected: 169")
assert X_full.shape[1] == 169, (
    f"Feature count mismatch: got {X_full.shape[1]}, expected 169"
)
print("Feature count = 169")

categories = {
    "raw_numeric": 0,
    "lag_value_diff": 0,
    "lag_value_ratio": 0,
    "ClusterNo": 0,
    "Residual_savgol_w5p3": 0,
    "dayofyear": 0,
}
for col in X_full.columns:
    if col.startswith("lag_value_ratio_"):
        categories["lag_value_ratio"] += 1
    elif col.startswith("lag_value_"):
        categories["lag_value_diff"] += 1
    elif col == "ClusterNo":
        categories["ClusterNo"] += 1
    elif col == "Residual_savgol_w5p3":
        categories["Residual_savgol_w5p3"] += 1
    elif col == "dayofyear":
        categories["dayofyear"] += 1
    else:
        categories["raw_numeric"] += 1

print()
print("Feature category breakdown:")
for cat, count in categories.items():
    print(f"  {cat}: {count}")
print(f"  Total: {sum(categories.values())}")

Feature count: 169
Expected: 169
Feature count = 169

Feature category breakdown:
  raw_numeric: 46
  lag_value_diff: 60
  lag_value_ratio: 60
  ClusterNo: 1
  Residual_savgol_w5p3: 1
  dayofyear: 1
  Total: 169


In [7]:
print("NaN rate per feature class:")
class_specs = [
    (
        "raw_numeric",
        lambda c: not c.startswith("lag_value_")
        and c not in ["ClusterNo", "Residual_savgol_w5p3", "dayofyear"],
    ),
    (
        "lag_value_diff",
        lambda c: c.startswith("lag_value_") and not c.startswith("lag_value_ratio_"),
    ),
    ("lag_value_ratio", lambda c: c.startswith("lag_value_ratio_")),
    ("ClusterNo", lambda c: c == "ClusterNo"),
    ("Residual_savgol_w5p3", lambda c: c == "Residual_savgol_w5p3"),
    ("dayofyear", lambda c: c == "dayofyear"),
]
for name, fn in class_specs:
    cols = [c for c in X_full.columns if fn(c)]
    if not cols:
        continue
    total = len(cols) * len(X_full)
    nans = X_full[cols].isna().sum().sum()
    print(f"  {name:25s} {len(cols):4d} cols  NaN: {nans / total * 100:5.2f}%")

NaN rate per feature class:
  raw_numeric                 46 cols  NaN:  0.13%
  lag_value_diff              60 cols  NaN:  7.12%


  lag_value_ratio             60 cols  NaN:  7.12%
  ClusterNo                    1 cols  NaN:  0.00%
  Residual_savgol_w5p3         1 cols  NaN:  6.15%
  dayofyear                    1 cols  NaN:  0.00%


In [8]:
neg = train[train["anomaly"] == 0]
pos = train[train["anomaly"] == 1]
print(f"Pre-downsampling: {len(neg):,} normal + {len(pos):,} anomaly")

negs1 = neg.sample(n=pos.shape[0], random_state=10)
negs2 = neg.sample(n=pos.shape[0], random_state=20)
df_eq = pd.concat([negs1, pos, negs2, pos], axis=0).reset_index(drop=True)

print(f"Post-downsampling: {df_eq.shape[0]:,} rows")
print(
    f"Class balance: {(df_eq['anomaly'] == 0).sum():,} : {(df_eq['anomaly'] == 1).sum():,}"
)

Pre-downsampling: 1,712,198 normal + 37,296 anomaly


Post-downsampling: 149,184 rows
Class balance: 74,592 : 74,592


In [9]:
X_eq = df_eq.select_dtypes(include=["float", "int"])
X_eq = X_eq.drop(columns=[c for c in drop_cols if c in X_eq.columns])
print(f"X_eq features: {X_eq.shape[1]} (expected 169)")
assert X_eq.shape[1] == 169

train_mask = df_eq["building_id"] % 5 < 4
val_mask = df_eq["building_id"] % 5 == 4

X_train = X_eq[train_mask]
y_train = df_eq.loc[train_mask, "anomaly"]
X_val = X_eq[val_mask]
y_val = df_eq.loc[val_mask, "anomaly"]

print(f"Train: {X_train.shape}, {df_eq[train_mask]['building_id'].nunique()} buildings")
print(f"Val: {X_val.shape}, {df_eq[val_mask]['building_id'].nunique()} buildings")

X_eq features: 169 (expected 169)
Train: (119613, 169), 162 buildings
Val: (29571, 169), 38 buildings


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print(f"X_train_scaled: {X_train_scaled.shape}, NaN={np.isnan(X_train_scaled).sum():,}")
print(f"X_val_scaled:   {X_val_scaled.shape}, NaN={np.isnan(X_val_scaled).sum():,}")

X_train_scaled: (119613, 169), NaN=696,686
X_val_scaled:   (29571, 169), NaN=247,912


In [11]:
model = lgb.LGBMClassifier(n_estimators=100, verbose=-1, random_state=42)
model.fit(X_train_scaled, y_train)

preds = model.predict_proba(X_val_scaled)[:, 1]
val_auc = roc_auc_score(y_val, preds)

print(f"{'=' * 60}")
print(f"M2.2.e LightGBM val AUC: {val_auc:.4f}")
print(f"{'=' * 60}")

# M2.3 aliases
pred_lgb = preds
auc_lgb = val_auc
model_lgb = model

M2.2.e LightGBM val AUC: 0.9818


C:\Users\tonykuo\projects\lead-reproduction\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [12]:
baseline_auc = 0.8952  # M2.1
paper_target = 0.9849  # paper Table 2
delta_auc = val_auc - baseline_auc

print(f"M2.1 baseline (57 features):    {baseline_auc:.4f}")
print(f"M2.2.e (169 features):          {val_auc:.4f}")
print(f"delta_AUC vs M2.1:              {delta_auc:+.4f}")
print(f"Paper Table 2 target:           {paper_target:.4f}")
print(f"Gap vs paper:                   {(paper_target - val_auc) * 100:+.2f}%")
print()
print("Done when criteria:")
print(f"  val AUC >= 0.97:  {'pass' if val_auc >= 0.97 else 'FAIL'}")
print(f"  delta_AUC >= +0.04: {'pass' if delta_auc >= 0.04 else 'FAIL'}")

M2.1 baseline (57 features):    0.8952
M2.2.e (169 features):          0.9818
delta_AUC vs M2.1:              +0.0866
Paper Table 2 target:           0.9849
Gap vs paper:                   +0.31%

Done when criteria:
  val AUC >= 0.97:  pass
  delta_AUC >= +0.04: pass


In [13]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
top10 = importances.nlargest(10)

print("Our top 10 importance:")
for i, (feat, imp) in enumerate(top10.items(), 1):
    print(f"  {i:2d}. {feat:35s} {imp:6.0f}")

paper_top10 = [
    "building_id",
    "lag_value_ratio_1",
    "lag_value_ratio_-1",
    "meter_reading",
    "dayofyear",
    "square_feet",
    "gte_building_id",
    "lag_value_ratio_-168",
    "lag_value_ratio_2",
    "gte_meter_primary_use",
]
print()
print("Paper Fig 5 top 10 (our name mapping):")
for i, feat in enumerate(paper_top10, 1):
    marker = "in ours" if feat in top10.index else "-"
    print(f"  {i:2d}. {feat:35s} {marker}")

overlap = set(paper_top10) & set(top10.index)
print(f"Overlap: {len(overlap)}/10")

Our top 10 importance:
   1. building_id                            206
   2. lag_value_ratio_1                      132
   3. meter_reading                          128
   4. lag_value_ratio_-1                     118
   5. dayofyear                              113
   6. Residual_savgol_w5p3                   105
   7. square_feet                             66
   8. lag_value_ratio_-168                    59
   9. lag_value_ratio_2                       56
  10. lag_value_ratio_168                     56

Paper Fig 5 top 10 (our name mapping):
   1. building_id                         in ours
   2. lag_value_ratio_1                   in ours
   3. lag_value_ratio_-1                  in ours
   4. meter_reading                       in ours
   5. dayofyear                           in ours
   6. square_feet                         in ours
   7. gte_building_id                     -
   8. lag_value_ratio_-168                in ours
   9. lag_value_ratio_2                   in ours
  1

In [14]:
import xgboost as xgb
import time

print("Training XGBoost (n_estimators=100)...")
t0 = time.time()
model_xgb = xgb.XGBClassifier(
    n_estimators=100, eval_metric="logloss", verbosity=0, random_state=42
)
model_xgb.fit(X_train_scaled, y_train)
elapsed_xgb = time.time() - t0
print(f"XGBoost trained in {elapsed_xgb:.1f}s")

pred_xgb = model_xgb.predict_proba(X_val_scaled)[:, 1]
auc_xgb = roc_auc_score(y_val, pred_xgb)
print(f"XGBoost val AUC: {auc_xgb:.4f}")

Training XGBoost (n_estimators=100)...


XGBoost trained in 2.4s
XGBoost val AUC: 0.9749


In [15]:
from catboost import CatBoostClassifier

print("Training CatBoost (iterations=1000, may take 5-15 min on CPU)...")
t0 = time.time()
model_cat = CatBoostClassifier(iterations=1000, verbose=False, random_seed=42)
model_cat.fit(X_train_scaled, y_train)
elapsed_cat = time.time() - t0
print(f"CatBoost trained in {elapsed_cat / 60:.1f} min")

pred_cat = model_cat.predict_proba(X_val_scaled)[:, 1]
auc_cat = roc_auc_score(y_val, pred_cat)
print(f"CatBoost val AUC: {auc_cat:.4f}")

Training CatBoost (iterations=1000, may take 5-15 min on CPU)...


CatBoost trained in 0.2 min
CatBoost val AUC: 0.9797


In [16]:
# CatBoost iteration verification
print("CatBoost configured iterations: 1000")
print(f"CatBoost actual tree_count_:    {model_cat.tree_count_}")
print(f"CatBoost best_iteration_:       {model_cat.best_iteration_}")

if model_cat.tree_count_ < 1000:
    print(f"\n⚠️  CatBoost only ran {model_cat.tree_count_} iterations (out of 1000)")
    print("   Likely cause: built-in overfitting detector triggered early stop")
    print("   This may NOT match buds-lab's exact run (which ran full 1000 iter)")
else:
    print("\n✓ CatBoost ran full 1000 iterations as configured")

CatBoost configured iterations: 1000
CatBoost actual tree_count_:    1000
CatBoost best_iteration_:       None

✓ CatBoost ran full 1000 iterations as configured


In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier

# HistGBT does not support NaN in scaled data; fill with 0
X_train_filled = np.nan_to_num(X_train_scaled, nan=0)
X_val_filled = np.nan_to_num(X_val_scaled, nan=0)

print("Training HistGBT (max_iter=100)...")
t0 = time.time()
model_hist = HistGradientBoostingClassifier(max_iter=100, random_state=42)
model_hist.fit(X_train_filled, y_train)
elapsed_hist = time.time() - t0
print(f"HistGBT trained in {elapsed_hist:.1f}s")

pred_hist = model_hist.predict_proba(X_val_filled)[:, 1]
auc_hist = roc_auc_score(y_val, pred_hist)
print(f"HistGBT val AUC: {auc_hist:.4f}")

Training HistGBT (max_iter=100)...


HistGBT trained in 1.8s
HistGBT val AUC: 0.9806


In [18]:
# Equal-weight ensemble (per paper section 2.3.4 + Eq.4)
pred_ensemble = (pred_lgb + pred_xgb + pred_cat + pred_hist) / 4
auc_ensemble = roc_auc_score(y_val, pred_ensemble)

SEP = "=" * 60
sep = "-" * 50

print()
print(SEP)
print("M2.3 Results vs Paper Table 2")
print(SEP)
print(f"{'Model':<20s} {'Ours':>8s}   {'Paper':>8s}   {'Gap':>8s}")
print(sep)
print(f"{'LightGBM':<20s} {auc_lgb:.4f}    0.9849    {(0.9849 - auc_lgb) * 100:+.2f}%")
print(f"{'XGBoost':<20s} {auc_xgb:.4f}    0.9840    {(0.9840 - auc_xgb) * 100:+.2f}%")
print(f"{'CatBoost':<20s} {auc_cat:.4f}    0.9857    {(0.9857 - auc_cat) * 100:+.2f}%")
print(f"{'HistGBT':<20s} {auc_hist:.4f}    0.9839    {(0.9839 - auc_hist) * 100:+.2f}%")
print(sep)
print(
    f"{'Ensemble':<20s} {auc_ensemble:.4f}    0.9866    {(0.9866 - auc_ensemble) * 100:+.2f}%"
)
print()

cat_gt_lgb = auc_cat > auc_lgb
lgb_gt_xgb = auc_lgb > auc_xgb
xgb_gt_hist = auc_xgb > auc_hist
ranking_ok = cat_gt_lgb and lgb_gt_xgb and xgb_gt_hist
ensemble_best = auc_ensemble > max(auc_lgb, auc_xgb, auc_cat, auc_hist)
ensemble_pass = auc_ensemble >= 0.97

print("Done when:")
print(f"  Paper ranking Cat>LGB>XGB>HistGBT: {ranking_ok}")
print(
    f"    cat>lgb={cat_gt_lgb} ({auc_cat:.4f}>{auc_lgb:.4f}), lgb>xgb={lgb_gt_xgb}, xgb>hist={xgb_gt_hist}"
)
print(f"  Ensemble > max(individual): {ensemble_best}")
print(f"  Ensemble >= 0.97: {ensemble_pass}")


M2.3 Results vs Paper Table 2
Model                    Ours      Paper        Gap
--------------------------------------------------
LightGBM             0.9818    0.9849    +0.31%
XGBoost              0.9749    0.9840    +0.91%
CatBoost             0.9797    0.9857    +0.60%
HistGBT              0.9806    0.9839    +0.33%
--------------------------------------------------
Ensemble             0.9830    0.9866    +0.36%

Done when:
  Paper ranking Cat>LGB>XGB>HistGBT: False
    cat>lgb=False (0.9797>0.9818), lgb>xgb=True, xgb>hist=False
  Ensemble > max(individual): True
  Ensemble >= 0.97: True


In [19]:
# Cross-model feature importance comparison
# HistGBT (sklearn 1.7) does not expose feature_importances_; skipped for it.
import pandas as pd

model_lgb_name = "LightGBM"
models_with_imp = {
    "LightGBM": model_lgb,
    "XGBoost": model_xgb,
    "CatBoost": model_cat,
}

print("Top 5 feature importance per model (split-count / gain-based):")
print(f"{'Model':<12s}  Top 5 features")
print("-" * 80)

all_top10 = {}
for name, mdl in models_with_imp.items():
    imp = pd.Series(mdl.feature_importances_, index=X_train.columns)
    top5 = imp.nlargest(5).index.tolist()
    all_top10[name] = set(imp.nlargest(10).index)
    print(f"{name:<12s}  {top5}")

print()
print("Note: HistGBT skipped — sklearn 1.7 HistGradientBoostingClassifier")
print("      does not expose feature_importances_.")
print()

common_3 = set.intersection(*all_top10.values())
print("Features in all 3 models' top 10 (LGB / XGB / CatBoost):")
print(f"  Common: {sorted(common_3)}")
print(f"  Count:  {len(common_3)}/10")

print()
print("LGB-only top 10 (not in XGB or CatBoost):")
lgb_only = all_top10["LightGBM"] - all_top10["XGBoost"] - all_top10["CatBoost"]
print(f"  {sorted(lgb_only)}")

Top 5 feature importance per model (split-count / gain-based):
Model         Top 5 features
--------------------------------------------------------------------------------
LightGBM      ['building_id', 'lag_value_ratio_1', 'meter_reading', 'lag_value_ratio_-1', 'dayofyear']
XGBoost       ['meter_reading', 'lag_value_ratio_168', 'Residual_savgol_w5p3', 'lag_value_ratio_1', 'lag_value_ratio_19']
CatBoost      ['meter_reading', 'Residual_savgol_w5p3', 'building_id', 'lag_value_ratio_1', 'lag_value_ratio_-1']

Note: HistGBT skipped — sklearn 1.7 HistGradientBoostingClassifier
      does not expose feature_importances_.

Features in all 3 models' top 10 (LGB / XGB / CatBoost):
  Common: ['Residual_savgol_w5p3', 'lag_value_ratio_-1', 'lag_value_ratio_-168', 'lag_value_ratio_1', 'lag_value_ratio_168', 'meter_reading']
  Count:  6/10

LGB-only top 10 (not in XGB or CatBoost):
  ['lag_value_ratio_2']


In [20]:
# M2.4 Phase 1, Cell 18: Prepare val-aligned columns for post-processing
# df_eq[val_mask] rows correspond 1:1 with pred_ensemble (same val_mask used in Cell 9)
val_meter_reading = df_eq.loc[val_mask, "meter_reading"].values
val_dayofyear = df_eq.loc[val_mask, "dayofyear"].values
val_building_id = df_eq.loc[val_mask, "building_id"].values

assert len(val_meter_reading) == len(pred_ensemble), (
    f"Alignment mismatch: df_eq[val_mask] has {len(val_meter_reading)} rows "
    f"but pred_ensemble has {len(pred_ensemble)} rows"
)

print(f"Val rows: {len(val_meter_reading)}, pred_ensemble length: {len(pred_ensemble)}")
print("Alignment OK")
print()

# Pre-compute all three rule masks (reused in Cell 19 and Cell 20)
mask_rule1 = val_meter_reading == 1.0
mask_rule2a = (val_dayofyear == 1) & ((val_building_id > 145) | (val_building_id < 105))
mask_rule2b = val_dayofyear > 366.9583

print("Rule trigger counts on val set:")
print(f"  Rule 1  (meter_reading == 1.0):                {mask_rule1.sum():,}")
print(f"  Rule 2a (dayofyear==1 & bldg_id>145|<105):     {mask_rule2a.sum():,}")
print(f"  Rule 2b (dayofyear > 366.9583):                {mask_rule2b.sum():,}")
print()
print(f"val_dayofyear range:    [{val_dayofyear.min():.4f}, {val_dayofyear.max():.4f}]")
print(f"val_building_id range:  [{val_building_id.min()}, {val_building_id.max()}]")

Val rows: 29571, pred_ensemble length: 29571
Alignment OK

Rule trigger counts on val set:
  Rule 1  (meter_reading == 1.0):                6,528
  Rule 2a (dayofyear==1 & bldg_id>145|<105):     0
  Rule 2b (dayofyear > 366.9583):                2

val_dayofyear range:    [1.0417, 366.9583]
val_building_id range:  [69, 1319]


In [21]:
# M2.4 Phase 1, Cell 19: Apply 3 post-processing rules
# Order: Rule 1 first, Rule 2a/2b after (Rule 2 overwrites Rule 1)
# Source: buds-lab Modeling notebook Cell 14

pred_postproc = pred_ensemble.copy()

# Rule 1: meter_reading == 1.0 → definitively anomalous
pred_postproc[mask_rule1] = 1

# Rule 2a: dayofyear==1 AND (building_id > 145 OR < 105) → definitively normal
pred_postproc[mask_rule2a] = 0

# Rule 2b: dayofyear > 366.9583 → definitively normal
pred_postproc[mask_rule2b] = 0

auc_postproc = roc_auc_score(y_val, pred_postproc)
delta_pp = auc_postproc - auc_ensemble

print(f"Pre-postproc AUC:  {auc_ensemble:.4f}")
print(f"Post-postproc AUC: {auc_postproc:.4f}")
print(f"ΔAUC:              {delta_pp:+.4f}")
print()

if delta_pp > 0.01:
    print(
        "⚠️  ΔAUC > +0.01 — possible bug or Rule 2 range misinterpretation, stop and report"
    )
elif delta_pp < 0:
    print(
        "⚠️  ΔAUC < 0 — post-processing reduced AUC; rule direction may be wrong, stop and report"
    )
else:
    print("✓ ΔAUC in expected range (+0.001 ~ +0.005)")

Pre-postproc AUC:  0.9830
Post-postproc AUC: 0.9834
ΔAUC:              +0.0004

✓ ΔAUC in expected range (+0.001 ~ +0.005)


In [22]:
# M2.4 Phase 1, Cell 20: Per-rule isolated ΔAUC (Layer 3 finding)
# Apply each rule alone to measure its individual contribution

# Rule 1 only
pred_r1 = pred_ensemble.copy()
pred_r1[mask_rule1] = 1
auc_r1 = roc_auc_score(y_val, pred_r1)
delta_r1 = auc_r1 - auc_ensemble

# Rule 2a only
pred_r2a = pred_ensemble.copy()
pred_r2a[mask_rule2a] = 0
auc_r2a = roc_auc_score(y_val, pred_r2a)
delta_r2a = auc_r2a - auc_ensemble

# Rule 2b only
pred_r2b = pred_ensemble.copy()
pred_r2b[mask_rule2b] = 0
auc_r2b = roc_auc_score(y_val, pred_r2b)
delta_r2b = auc_r2b - auc_ensemble

print("Per-rule isolated ΔAUC vs ensemble baseline:")
print(
    f"  Rule 1  only (meter_reading==1 → 1):       AUC={auc_r1:.4f}  Δ={delta_r1:+.4f}"
)
print(
    f"  Rule 2a only (dayofyear==1 + bldg → 0):    AUC={auc_r2a:.4f}  Δ={delta_r2a:+.4f}"
)
print(
    f"  Rule 2b only (dayofyear>366.9583 → 0):     AUC={auc_r2b:.4f}  Δ={delta_r2b:+.4f}"
)
print(
    f"  All 3 rules combined:                      AUC={auc_postproc:.4f}  Δ={delta_pp:+.4f}"
)
print()

if delta_r1 < delta_r2a or delta_r1 < delta_r2b:
    print("⚠️  Rule 1 ΔAUC is NOT the largest — unexpected, stop and report")
else:
    print("✓ Rule 1 ΔAUC is largest as expected")

for rule_name, d in [("Rule 2a", delta_r2a), ("Rule 2b", delta_r2b)]:
    if d < 0:
        print(f"⚠️  {rule_name} ΔAUC < 0 when applied alone — stop and report")

Per-rule isolated ΔAUC vs ensemble baseline:
  Rule 1  only (meter_reading==1 → 1):       AUC=0.9834  Δ=+0.0004
  Rule 2a only (dayofyear==1 + bldg → 0):    AUC=0.9830  Δ=+0.0000
  Rule 2b only (dayofyear>366.9583 → 0):     AUC=0.9830  Δ=+0.0000
  All 3 rules combined:                      AUC=0.9834  Δ=+0.0004

✓ Rule 1 ΔAUC is largest as expected


In [23]:
# M2.4 Phase 1, Cell 21: Confusion matrix at threshold=0.5; compare with paper Fig 3
from sklearn.metrics import confusion_matrix

threshold = 0.5
y_pred_bin = (pred_postproc >= threshold).astype(int)

cm = confusion_matrix(y_val, y_pred_bin)
TN, FP, FN, TP = cm.ravel()
total = len(y_val)

print("Confusion Matrix (threshold=0.5, post-postproc predictions):")
print(f"  Actual=0: TN={TN:,}  FP={FP:,}")
print(f"  Actual=1: FN={FN:,}   TP={TP:,}")
print()
print("As % of total predictions:")
print(f"  TN: {TN / total * 100:.1f}%    FP: {FP / total * 100:.1f}%")
print(f"  FN: {FN / total * 100:.1f}%    TP: {TP / total * 100:.1f}%")
print()
print("Paper Fig 3 reference (% of total):")
print("  TN: 96.3%   FP: 1.4%")
print("  FN: 0.2%    TP: 2.0%")
print()

# Two precision/recall computations — paper §3 text vs Fig 3 back-calc
precision_std = TP / (TP + FP) if (TP + FP) > 0 else 0
recall_std = TP / (TP + FN) if (TP + FN) > 0 else 0

# Fig 3 back-calc: TP=2.0%, FP=1.4%, FN=0.2%
# → precision = 2.0/(2.0+1.4) = 58.8%, recall = 2.0/(2.0+0.2) = 90.9%
# Paper §3 text: precision=98.7%, recall=81.9%
print("Precision/Recall (our numbers):")
print(f"  Precision (TP/(TP+FP)): {precision_std * 100:.1f}%")
print(f"  Recall    (TP/(TP+FN)): {recall_std * 100:.1f}%")
print()
print("Paper §3 text:              precision=98.7%, recall=81.9%")
print("Paper Fig 3 back-calc:      precision=58.8%, recall=90.9%")
print("(known paper inconsistency: §3 text vs Fig 3 numbers do not reconcile)")

Confusion Matrix (threshold=0.5, post-postproc predictions):
  Actual=0: TN=13,875  FP=162
  Actual=1: FN=2,926   TP=12,608

As % of total predictions:
  TN: 46.9%    FP: 0.5%
  FN: 9.9%    TP: 42.6%

Paper Fig 3 reference (% of total):
  TN: 96.3%   FP: 1.4%
  FN: 0.2%    TP: 2.0%

Precision/Recall (our numbers):
  Precision (TP/(TP+FP)): 98.7%
  Recall    (TP/(TP+FN)): 81.2%

Paper §3 text:              precision=98.7%, recall=81.9%
Paper Fig 3 back-calc:      precision=58.8%, recall=90.9%
(known paper inconsistency: §3 text vs Fig 3 numbers do not reconcile)


In [ ]:
# M2.4 Phase 1, Cell 22: Summary
SEP = "=" * 60
noise_floor = 0.0005

print(SEP)
print("M2.4 Phase 1 Summary: Val Side Post-processing")
print(SEP)
print()
print("AUC:")
print(f"  Pre-postproc ensemble AUC:   {auc_ensemble:.4f}  (M2.3)")
print(f"  Post-postproc ensemble AUC:  {auc_postproc:.4f}")
print(f"  ΔAUC (3 rules combined):     {delta_pp:+.4f}")
significant = abs(delta_pp) > noise_floor
print(
    f"  Significant (> ±{noise_floor}):      {'yes' if significant else 'no (within noise floor)'}"
)
print()
print("Per-rule isolated ΔAUC:")
print(f"  Rule 1  (meter_reading==1):  {delta_r1:+.4f}")
print(f"  Rule 2a (dayofyear==1+bldg): {delta_r2a:+.4f}")
print(f"  Rule 2b (dayofyear>366.958): {delta_r2b:+.4f}")
print()
print("Confusion matrix (threshold=0.5, post-postproc):")
print(f"  TN={TN:,} ({TN / total * 100:.1f}%)  FP={FP:,} ({FP / total * 100:.1f}%)")
print(f"  FN={FN:,}  ({FN / total * 100:.1f}%)   TP={TP:,}  ({TP / total * 100:.1f}%)")
print()
print("Precision/Recall vs paper:")
print(f"  Ours:              {precision_std * 100:.1f}% / {recall_std * 100:.1f}%")
print("  Paper §3 text:     98.7% / 81.9%")
print("  Paper Fig 3 calc:  58.8% / 90.9%")
print()
print("Done when:")
print("  [✓] Cell 18 alignment (no IndexError)")
print("  [✓] Cell 19 3-rule ΔAUC printed")
print("  [✓] Cell 20 per-rule ΔAUC printed")
print("  [✓] Cell 21 confusion matrix + paper Fig 3 comparison")
print("  [✓] Cell 22 summary complete")

In [ ]:
# M2.4 Phase 2, Cell 23: Test feature engineering
# Mirrors train pipeline (Cells 1-9) but uses timestamp-merge for value-change (unknown #11)
import datetime
import time

# Step 1: Load test (57 cols including row_id)
test_raw = pd.read_csv("../data/raw/test_features.csv")
print(f"Test loaded: {test_raw.shape}")
assert test_raw.shape[0] == 1_800_567, f"Expected 1,800,567, got {test_raw.shape[0]}"

# Step 2: cloud_coverage fix (M2.2.0)
test_raw["cloud_coverage"] = test_raw["cloud_coverage"].replace({255: 10})

# Step 3: ClusterNo merge (M2.2.a) — same cluster labels, no re-fit needed
clusterno_df = pd.read_csv("../data/interim/clusterno.csv")
test_raw = test_raw.merge(clusterno_df, on="building_id", how="left")
assert test_raw["ClusterNo"].isna().sum() == 0, "Some test buildings missing ClusterNo"
print(f"After ClusterNo: {test_raw.shape}")

# Step 4: Value-change features — timestamp-merge (per buds-lab test Cells 11+12)
# One loop of 60 merges → both diff and ratio (vs buds-lab's two loops of 60)
test_feature = pd.read_csv("../data/raw/test_features.csv")  # lookup source

shifts = (
    list(np.arange(-24, 0).astype(int))
    + list(np.arange(1, 25).astype(int))
    + list(np.arange(-168, -24, 24).astype(int))
    + list(np.arange(48, 169, 24).astype(int))
)
assert len(shifts) == 60, f"Expected 60 shifts, got {len(shifts)}"

t0 = time.time()
for i, sh in enumerate(shifts):
    sh = int(sh)  # numpy.int64 → Python int (required for datetime.timedelta)
    meter_shift = test_feature[["building_id", "timestamp", "meter_reading"]].copy()
    meter_shift["timestamp"] = (
        pd.to_datetime(meter_shift["timestamp"]) + datetime.timedelta(hours=sh)
    ).astype(str)
    meter_shift = meter_shift.rename(columns={"meter_reading": "shifted_reading"})
    merged = test_feature.merge(
        meter_shift, on=["building_id", "timestamp"], how="left"
    )
    test_raw[f"lag_value_{sh}"] = merged["shifted_reading"] - merged["meter_reading"]
    test_raw[f"lag_value_ratio_{sh}"] = (merged["shifted_reading"] + 1) / (
        merged["meter_reading"] + 1
    )
    if (i + 1) % 12 == 0:
        print(f"  {i + 1}/60 shifts ({time.time() - t0:.0f}s)")
del merged, meter_shift, test_feature
lag_cols_test = [c for c in test_raw.columns if c.startswith("lag_value_")]
assert len(lag_cols_test) == 120, f"Expected 120, got {len(lag_cols_test)}"
print(f"Value-change done: {time.time() - t0:.1f}s total")

# Step 5: SavGol residual (M2.2.c)
results = []
for bid in test_raw["building_id"].unique():
    tmp = test_raw[test_raw["building_id"] == bid].copy()
    smoothed = savgol_filter(tmp["meter_reading"].ffill().bfill().fillna(0), 5, 3)
    tmp["Residual_savgol_w5p3"] = tmp["meter_reading"] - smoothed
    results.append(tmp)
test_raw = pd.concat(results).sort_index()
del results
print(f"After SavGol: {test_raw.shape}")

# Step 6: dayofyear (M2.2.d)
test_raw["dayofyear"] = (
    pd.to_datetime(test_raw["timestamp"]).dt.dayofyear
    + pd.to_datetime(test_raw["timestamp"]).dt.hour / 24
)

# Verify 169 features present and selectable
feature_cols = X_eq.columns.tolist()
assert len(feature_cols) == 169
check = test_raw[feature_cols]
assert check.shape == (1_800_567, 169), f"Unexpected shape: {check.shape}"
del check
print("Test feature verification: 1,800,567 rows × 169 features OK")
print(f"test_raw.shape = {test_raw.shape}")
print(
    f"dayofyear range: [{test_raw['dayofyear'].min():.4f}, {test_raw['dayofyear'].max():.4f}]"
)

In [26]:
# M2.4 Phase 2, Cell 24: Refit 4 models on X_all (full downsampled, no val split)
# Per unknown #16: X_all = scaler fitted on all df_eq (not just train split)
import time

feature_cols = X_eq.columns.tolist()  # 169 features
features_all = df_eq[feature_cols].copy()
target_all = df_eq["anomaly"]

assert len(features_all) == len(target_all)
print(f"features_all: {features_all.shape}, target_all: {len(target_all)}")

# Scaler fitted on ALL downsampled data (not just train split)
scaler_all = StandardScaler()
X_all = scaler_all.fit_transform(features_all)
X_test_scaled = scaler_all.transform(test_raw[feature_cols])

print(f"X_all: {X_all.shape}, NaN={np.isnan(X_all).sum():,}")
print(f"X_test_scaled: {X_test_scaled.shape}, NaN={np.isnan(X_test_scaled).sum():,}")

print()
t0 = time.time()
print("Refitting LightGBM on X_all...")
lgb_all = lgb.LGBMClassifier(n_estimators=100, verbose=-1, random_state=42)
lgb_all.fit(X_all, target_all)
print(f"  Done: {time.time() - t0:.1f}s")

t0 = time.time()
print("Refitting XGBoost on X_all...")
xgb_all = xgb.XGBClassifier(
    n_estimators=100, eval_metric="logloss", verbosity=0, random_state=42
)
xgb_all.fit(X_all, target_all)
print(f"  Done: {time.time() - t0:.1f}s")

t0 = time.time()
print("Refitting CatBoost on X_all (1000 iterations)...")
cat_all = CatBoostClassifier(iterations=1000, verbose=False, random_seed=42)
cat_all.fit(X_all, target_all)
print(f"  Done: {time.time() - t0:.1f}s")

t0 = time.time()
print("Refitting HistGBT on X_all...")
hist_all = HistGradientBoostingClassifier(max_iter=100, random_state=42)
hist_all.fit(np.nan_to_num(X_all), target_all)
print(f"  Done: {time.time() - t0:.1f}s")

print("\nAll 4 models refitted on X_all")

features_all: (149184, 169), target_all: 149184


X_all: (149184, 169), NaN=944,598


X_test_scaled: (1800567, 169), NaN=13,876,776

Refitting LightGBM on X_all...


  Done: 1.5s
Refitting XGBoost on X_all...


  Done: 2.0s
Refitting CatBoost on X_all (1000 iterations)...


  Done: 18.3s
Refitting HistGBT on X_all...


  Done: 2.1s

All 4 models refitted on X_all


In [27]:
# M2.4 Phase 2, Cell 25: Predict test set + ensemble
print("Predicting test set (1,800,567 rows)...")
pred_lgb_test = lgb_all.predict_proba(X_test_scaled)[:, 1]
pred_xgb_test = xgb_all.predict_proba(X_test_scaled)[:, 1]
pred_cat_test = cat_all.predict_proba(X_test_scaled)[:, 1]
pred_hist_test = hist_all.predict_proba(np.nan_to_num(X_test_scaled))[:, 1]

pred_ensemble_test = (
    pred_lgb_test + pred_xgb_test + pred_cat_test + pred_hist_test
) / 4

assert len(pred_ensemble_test) == 1_800_567, (
    f"Expected 1,800,567, got {len(pred_ensemble_test)}"
)
assert pred_ensemble_test.min() >= 0 and pred_ensemble_test.max() <= 1, (
    "Predictions out of [0,1]"
)

print(f"Ensemble test predictions: {len(pred_ensemble_test):,}")
print(f"Range: [{pred_ensemble_test.min():.4f}, {pred_ensemble_test.max():.4f}]")
print(f"Mean:  {pred_ensemble_test.mean():.4f}")

Predicting test set (1,800,567 rows)...


C:\Users\tonykuo\projects\lead-reproduction\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Ensemble test predictions: 1,800,567
Range: [0.0005, 0.9999]
Mean:  0.0738


In [28]:
# M2.4 Phase 2, Cell 26: Apply 3 post-processing rules on test set
# Same rules as val (Cell 19), applied to test predictions
pred_test_postproc = pred_ensemble_test.copy()

# Rule 1: meter_reading == 1.0 → anomaly = 1
mask_r1_test = test_raw["meter_reading"].values == 1.0
pred_test_postproc[mask_r1_test] = 1
print(f"Rule 1 triggered: {mask_r1_test.sum():,} rows (meter_reading==1.0)")

# Rule 2a: dayofyear==1 AND (building_id > 145 OR < 105) → anomaly = 0
mask_r2a_test = (test_raw["dayofyear"].values == 1) & (
    (test_raw["building_id"].values > 145) | (test_raw["building_id"].values < 105)
)
pred_test_postproc[mask_r2a_test] = 0
print(
    f"Rule 2a triggered: {mask_r2a_test.sum():,} rows (dayofyear==1 + bldg_id filter)"
)

# Rule 2b: dayofyear > 366.9583 → anomaly = 0
mask_r2b_test = test_raw["dayofyear"].values > 366.9583
pred_test_postproc[mask_r2b_test] = 0
print(f"Rule 2b triggered: {mask_r2b_test.sum():,} rows (dayofyear>366.9583)")
print()
print(f"Post-processed test predictions: {len(pred_test_postproc):,}")

if mask_r2a_test.sum() == 0:
    print("⚠️  Rule 2a triggered 0 rows on test — stop and report to Tony")
elif mask_r2b_test.sum() == 0:
    print("⚠️  Rule 2b triggered 0 rows on test — stop and report to Tony")
else:
    print("✓ Rule 2a and Rule 2b both triggered on test")

Rule 1 triggered: 17,660 rows (meter_reading==1.0)
Rule 2a triggered: 192 rows (dayofyear==1 + bldg_id filter)
Rule 2b triggered: 206 rows (dayofyear>366.9583)

Post-processed test predictions: 1,800,567
✓ Rule 2a and Rule 2b both triggered on test


In [29]:
# M2.4 Phase 2, Cell 27: Save submission CSV
import os

assert "row_id" in test_raw.columns, "test_raw missing row_id"

submission = pd.DataFrame(
    {
        "row_id": test_raw["row_id"].values,
        "anomaly": pred_test_postproc,
    }
)

assert len(submission) == 1_800_567, f"Expected 1,800,567 rows, got {len(submission)}"

os.makedirs("../data/processed", exist_ok=True)
submission_path = "../data/processed/submission_m2_4_phase2.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission saved: {submission_path}")
print(f"Rows: {len(submission):,}")
print()
print("Anomaly distribution:")
print(submission["anomaly"].describe())

Submission saved: ../data/processed/submission_m2_4_phase2.csv
Rows: 1,800,567

Anomaly distribution:
count    1.800567e+06
mean     7.373943e-02
std      1.647245e-01
min      0.000000e+00
25%      7.363850e-03
50%      1.951915e-02
75%      5.622705e-02
max      1.000000e+00
Name: anomaly, dtype: float64
